## read data

In [6]:
import pandas as pd

raw_path = "../data/raw/ilc_di01_income_quintiles.tsv"
df_raw = pd.read_csv(
    raw_path,
    sep="\t",
    na_values=[":", ": "],  # treat ':' as missing
)

df_raw.to_parquet("../data/clean/ilc_di01_income_quintiles.parquet", index=False)

## code

In [7]:
import pandas as pd

# Load your Parquet file
df = pd.read_parquet("../data/clean/ilc_di01_income_quintiles.parquet")

In [8]:
df.head

<bound method NDFrame.head of      freq,quantile,indic_il,currency,geo\TIME_PERIOD 1995  1996  1997  1998   \
0                                  A,D1,SHARE,EUR,AL  None  None  None  None   
1                                  A,D1,SHARE,EUR,AT    3     4     4     4    
2                                  A,D1,SHARE,EUR,BE    3     3     4     4    
3                                  A,D1,SHARE,EUR,BG  None  None  None  None   
4                                  A,D1,SHARE,EUR,CH  None  None  None  None   
...                                              ...   ...   ...   ...   ...   
7241                              A,QU5,SHARE,PPS,SI  None  None  None  None   
7242                              A,QU5,SHARE,PPS,SK  None  None  None  None   
7243                              A,QU5,SHARE,PPS,TR  None  None  None  None   
7244                              A,QU5,SHARE,PPS,UK   40    40    38    39    
7245                              A,QU5,SHARE,PPS,XK  None  None  None  None   

     1999

In [11]:
import pandas as pd

# Load your Parquet file
df = pd.read_parquet("../data/clean/ilc_di01_income_quintiles.parquet")

# Take the first column (string with comma-separated codes)
first_col = df.columns[0]

# Split into separate columns
split_cols = df[first_col].str.split(",", expand=True)

# Give them names (adjust as needed)
split_cols.columns = ["freq", "quantile", "indic_il", "currency", "geo"]  # or whatever labels you need

# Drop the original combined column and join the new ones
df = pd.concat([split_cols, df.drop(columns=[first_col])], axis=1)



year_cols = [c for c in df.columns if c.strip().isdigit()]


for col in year_cols:
    df.loc[:, col] = (
        df[col]
        .astype(str)                # make everything string
        .str.extract(r"([\d.]+)")[0]       # keep only number part like 28.5
        .astype(float)                     # to float
    )

df.head()

,freq,quantile,indic_il,currency,geo,1995,1996,1997,1998,1999,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,D1,SHARE,EUR,AL,NaN,NaN,NaN,NaN,NaN,...,NaN,2.0,2.0,2.3,2.5,2.7,3.0,3.0,NaN,NaN
1,A,D1,SHARE,EUR,AT,3.0,4.0,4.0,4.0,4.0,...,3.3,3.1,3.4,3.2,3.3,3.2,3.1,3.1,3.1,NaN
2,A,D1,SHARE,EUR,BE,3.0,3.0,4.0,4.0,4.0,...,3.7,3.6,3.7,3.9,3.9,4.1,4.0,4.2,4.1,4.3
3,A,D1,SHARE,EUR,BG,NaN,NaN,NaN,NaN,NaN,...,1.8,1.9,2.1,2.1,2.1,2.3,2.2,2.4,2.4,2.3
4,A,D1,SHARE,EUR,CH,NaN,NaN,NaN,NaN,NaN,...,3.4,3.2,3.3,3.1,3.1,3.1,3.2,3.1,3.2,NaN


In [ ]:
year_cols = [c for c in df.columns if c.strip().isdigit()]

df_long = df.melt(
    id_vars=["freq", "quantile", "indic_il", "currency", "geo"],
    value_vars=year_cols,
    var_name="year",
    value_name="value"
)

df_long["year"] = df_long["year"].astype(int)
df_long = df_long.dropna(subset=["value"])

KeyError: "The following id_vars or value_vars are not present in the DataFrame: ['age', 'unit']"

In [8]:
df_long.head()

,country,indicator,year,value
1,AT,GINI_HND,2014,27.6
2,BE,GINI_HND,2014,25.9
3,BG,GINI_HND,2014,35.4
4,CH,GINI_HND,2014,29.5
5,CY,GINI_HND,2014,34.8


In [9]:
df_long.to_parquet("../data/processed/stg_gini.parquet", index=False)